# U-Net · LGG MRI Brain Tumor Segmentation
**Deep Learning Assignment — Medical Image Segmentation**

---
### Steps
1. Check GPU
2. Install dependencies
3. Mount Google Drive & unzip dataset
4. Verify dataset structure
5. Train the model
6. Evaluate & visualize results
7. Start API server for frontend


---
## Step 1 — Check GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

---
## Step 2 — Install Dependencies

In [ ]:
!pip install -q \
    segmentation-models-pytorch \
    albumentations \
    flask \
    flask-cors \
    flask-ngrok2 \
    pyngrok \
    pyyaml

print('All dependencies installed.')

---
## Step 3 — Mount Google Drive & Unzip Dataset

Upload your dataset zip to Google Drive first, then run this cell.  
Update `ZIP_PATH` to match where you saved it.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── UPDATE THIS PATH ──────────────────────────────────────────
ZIP_PATH  = '/content/drive/MyDrive/lgg-mri-segmentation.zip'
DEST_DIR  = '/content/data'
# ─────────────────────────────────────────────────────────────

os.makedirs(DEST_DIR, exist_ok=True)
!unzip -q "{ZIP_PATH}" -d "{DEST_DIR}"

print('Unzip complete. Contents:')
for entry in os.listdir(DEST_DIR):
    print(f'  {DEST_DIR}/{entry}')

---
## Step 4 — Verify Dataset Structure

In [ ]:
import os

DATA_ROOT = '/content/data/kaggle_3m'

image_paths, mask_paths = [], []

for root, _, files in os.walk(DATA_ROOT):
    for fname in sorted(files):
        if not fname.endswith('.tif') or fname.endswith('_mask.tif'):
            continue
        name      = os.path.splitext(fname)[0]
        mask_path = os.path.join(root, f'{name}_mask.tif')
        if os.path.exists(mask_path):
            image_paths.append(os.path.join(root, fname))
            mask_paths.append(mask_path)

print(f'Image-mask pairs found : {len(image_paths)}')
print(f'Sample image : {image_paths[0]}')
print(f'Sample mask  : {mask_paths[0]}')

# Quick sanity — display one pair
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img  = np.array(Image.open(image_paths[0]).convert('RGB'))
mask = np.array(Image.open(mask_paths[0]).convert('L'))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img);  axes[0].set_title('MRI Image'); axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Mask'); axes[1].axis('off')
plt.tight_layout()
plt.show()

---
## Step 5 — Upload Project Files from Your Machine

Upload `train.py` and `predict.py` using the cell below,  
**or** skip this if you already placed them in Google Drive.

In [ ]:
from google.colab import files

print('Select train.py and predict.py from your local machine:')
uploaded = files.upload()

for fname in uploaded:
    print(f'Uploaded: {fname}')

---
## Step 6 — Train the Model

In [ ]:
# ── CONFIG — adjust as needed ─────────────────
DATA_ROOT        = '/content/data/kaggle_3m'
MODEL_SAVE_PATH  = '/content/models/unet_best.pth'
EPOCHS           = 50
BATCH_SIZE       = 8
IMAGE_SIZE       = 256
LR               = 1e-4
# ─────────────────────────────────────────────

!python train.py \
    --data_root        "{DATA_ROOT}" \
    --model_save_path  "{MODEL_SAVE_PATH}" \
    --epochs           {EPOCHS} \
    --batch_size       {BATCH_SIZE} \
    --image_size       {IMAGE_SIZE} \
    --lr               {LR}

---
## Step 7 — View Training History

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('logs/training_history.png')
plt.figure(figsize=(15, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Step 8 — Run Inference on a Single Image

In [ ]:
import os

# Pick any image from the dataset
DATA_ROOT   = '/content/data/kaggle_3m'
MODEL_PATH  = '/content/models/unet_best.pth'
OUTPUT_PATH = '/content/output/prediction.png'

# Auto-pick the first available image
sample_image = None
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith('.tif') and not f.endswith('_mask.tif'):
            sample_image = os.path.join(root, f)
            break
    if sample_image:
        break

print(f'Running inference on: {sample_image}')

!python predict.py \
    --image_path  "{sample_image}" \
    --model_path  "{MODEL_PATH}" \
    --output_path "{OUTPUT_PATH}"

# Display result
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

result = mpimg.imread(OUTPUT_PATH)
plt.figure(figsize=(15, 5))
plt.imshow(result)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Step 9 — Save Model to Google Drive (Backup)

In [ ]:
import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/unet_mri_model'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy('/content/models/unet_best.pth', f'{DRIVE_SAVE_DIR}/unet_best.pth')
shutil.copy('logs/training_history.png',     f'{DRIVE_SAVE_DIR}/training_history.png')
shutil.copy('logs/config.yaml',              f'{DRIVE_SAVE_DIR}/config.yaml')

print(f'Model and logs saved to Google Drive → {DRIVE_SAVE_DIR}')

---
## Step 10 — Start API Server for Frontend

> **Run this last.** Copy the `https://...ngrok-free.app` URL printed below  
> and paste it into your frontend `.env` as `VITE_API_URL`.

In [ ]:
from pyngrok import ngrok
import subprocess, threading, time

MODEL_PATH = '/content/models/unet_best.pth'
PORT       = 5000

# Start Flask in a background thread
def run_server():
    subprocess.run([
        'python', 'predict.py',
        '--serve',
        '--model_path', MODEL_PATH,
        '--port', str(PORT),
        '--no_ngrok',               # We handle ngrok here manually
    ])

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)   # Wait for Flask to start

# Open ngrok tunnel
public_url = ngrok.connect(PORT)
print('=' * 50)
print(f'  API URL: {public_url}')
print('=' * 50)
print('Paste this URL into your frontend .env as VITE_API_URL')
print('Health check: ', public_url, '/health')

---
### Tips
- **Session disconnects?** Re-run Steps 2 and 10 only — model is saved to Drive.
- **ngrok URL changes** every session — update `.env` in your frontend each time.
- **Out of memory?** Reduce `BATCH_SIZE` to `4` in Step 6.
- **Slow training?** Make sure Runtime > Change runtime type is set to **T4 GPU**.